In [2]:
import pandas as pd
import numpy as np

# 데이터 불러오기
articles = pd.read_csv("data/articles_hm.csv")
customers = pd.read_csv("data/customer_hm.csv")
transactions = pd.read_csv("data/transactions_hm.csv")

articles_copy = articles.copy()
articles_copy

garment_mode = articles_copy[articles_copy['garment_group_name'] != 'Unknown']['garment_group_name'].mode()[0]
articles_copy['garment_group_name'] = articles_copy['garment_group_name'].replace('Unknown', garment_mode)

articles_copy[articles_copy['product_group_name'] =='Unknown']


customers_copy1 = customers.drop(columns = ["fashion_news_frequency"])
customers_copy1
customers_copy2 = customers_copy1.copy()
customers_copy2["age_group"] = "10대"
customers_copy2.loc[customers_copy2["age"] <= 120, "age_group"] = "70대+"
customers_copy2.loc[customers_copy2["age"] <= 69, "age_group"] = "60대"
customers_copy2.loc[customers_copy2["age"] <= 59, "age_group"] = "50대"
customers_copy2.loc[customers_copy2["age"] <= 49, "age_group"] = "40대"
customers_copy2.loc[customers_copy2["age"] <= 39, "age_group"] = "30대"
customers_copy2.loc[customers_copy2["age"] <= 29, "age_group"] = "20대"

customers_copy2["age_group"].value_counts()
# transactions 전처리를 위한 카피본 생성
transactions_copy1 = transactions.copy()
# t_dat 컬럼 전처리 전 형식 일치 확인
transactions_copy1[transactions_copy1["t_dat"].str.contains("-") == False]
# t_dat의 데이터형 문자열 -> 날짜데이터 변환
transactions_copy1["t_dat"] = pd.to_datetime(transactions_copy1["t_dat"])
transactions_copy1["t_dat"]
transactions_copy1["t_dat"] = transactions_copy1["t_dat"].dt.to_period("M")
transactions_copy1["t_dat"].value_counts()
# 1. product_code가 중복된 행들을 찾기 (keep=False를 써야 중복된 모든 행이 나옴)
is_duplicated_code = articles_copy.duplicated(subset=['product_code'], keep=False)

# 2. product_group_name이 Unknown인 행 찾기
is_unknown_group = (articles_copy['product_group_name'] == 'Unknown')

# 3. 두 조건을 모두 만족(&)하는 데이터 추출
target_df = articles_copy[is_duplicated_code & is_unknown_group]

# 4. 결과 확인
print(f"조건에 맞는 데이터 개수: {len(target_df)}개")
print(target_df[['article_id', 'product_code', 'prod_name', 'product_group_name']])
#2 테이블 조인 후 널값확인
tr_cus = transactions_copy1.merge(customers_copy2, on="customer_id", how="left")
tr_cus.isna().sum()
#3테이블 조인 후 널값확인
tr_cus_ar = tr_cus.merge(articles_copy, on="article_id", how="left")
tr_cus_ar.value_counts()

조건에 맞는 데이터 개수: 104개
        article_id  product_code             prod_name product_group_name
64       156224002        156224          Box 4p Socks            Unknown
4015     473954008        473954  OP Cheeky hipster 2p            Unknown
4016     473954013        473954  OP Cheeky hipster 2p            Unknown
4017     473954014        473954  OP Cheeky hipster 2p            Unknown
4018     473954015        473954  OP Cheeky hipster 2p            Unknown
...            ...           ...                   ...                ...
104298   921169001        921169    Brady Pull On Cord            Unknown
104299   921169002        921169    Brady Pull On Cord            Unknown
104622   925139001        925139             Ruben set            Unknown
104623   925139002        925139             Ruben set            Unknown
104624   925139003        925139             Ruben set            Unknown

[104 rows x 4 columns]


t_dat    customer_id                                                       article_id  price     sales_channel_id  FN   Active  club_member_status  age   age_group  product_code  prod_name               product_type_no  product_type_name  product_group_name  graphical_appearance_no  graphical_appearance_name  colour_group_code  colour_group_name  perceived_colour_value_id  perceived_colour_value_name  perceived_colour_master_id  perceived_colour_master_name  department_no  department_name  index_code  index_name          index_group_no  index_group_name  section_no  section_name                    garment_group_no  garment_group_name  detail_desc                                                                                                                                                                             
2019-02  94665b46e194622ccdbcadc0170f13a2f8ede1ff6d057d43a19b8938c808b662  629420001   0.008458  2                 0.0  0.0     ACTIVE              23.0  20대        629420 

In [3]:
tr_cus_ar.to_csv("data/cleaned_data.csv")